# 2. tokenizer2:

### Learning Objectives
By the end of this lesson, you should be able to:

* Build a production-ready BPE Tokenizer that correctly handles Unicode, whitespace normalization, and special tokens.
* Implement byte-level fallback so the Tokenizer can encode any input, including emojis, CJK text, and code, without generating unknown tokens.
* Use a pre-tokenization regex to split text at appropriate word, number, punctuation, and whitespace boundaries before executing BPE merges.
* Train a custom Tokenizer on a corpus and compare its compression ratio on multilingual text with `tiktoken`.
* Understand the role of Chat Templates in converting structured messages into Token IDs.
* Explain the differences between an educational Python implementation and a production-ready Tokenizer in terms of speed, accuracy, and reproducibility.
---
### What is the Problem?

The BPE Tokenizer from Lesson 01 worked on English text. Now test that same Tokenizer with Japanese text, emojis, or Python code containing a mix of tabs and spaces; part of the process will likely break or produce an unsuitable output.

The problem is not with the BPE algorithm itself; the problem is that the implementation is not yet complete. A production-ready Tokenizer must:

- Handle input at the byte level, independent of language;
- Normalize Unicode according to a defined policy before splitting text;
- Have special tokens that are never split or merged with other tokens;
- Combine pre-tokenization with subword splitting;
- Provide reliable, and ideally reversible, encode and decode capabilities;
- Be fast enough not to become a bottleneck in the training pipeline;
- Save vocabulary, merge rules, normalization, and special token configurations in a versioned format so results are reproducible.

The GPT-2 vocabulary contains 50,257 tokens, and Llama 3 uses a vocabulary of 128,256 tokens. For GPT-4 family models, tokenizers typically employ vocabularies on the scale of approximately 100,000 tokens; however, the exact number depends on the model and the encoding used.

These numbers do not belong to small toy examples. The merge tables for such vocabularies are trained on massive volumes of data. Beyond BPE itself, components such as normalization, pre-tokenization, special token handling, and chat template formatting separate a tokenizer limited to a "hello world" phrase from one suitable for extensive internet-scale data.

In this lesson, you will build and understand these very components and the logic behind them.

---

### Core Concept: The Full Pipeline

A production-ready Tokenizer is not just a single algorithm; it is a pipeline composed of several stages, each solving a different problem.

    A[Raw Text] --> B[Normalize] --> C[Pre-tokenize]--> D[BPE Merge]--> E[Special Tokens]--> F[Token IDs]





Packages

In [9]:
import re
import unicodedata
from collections import Counter
from typing import Dict, List, Tuple, Union

In [10]:
import regex

PATTERN = regex.compile(
    r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+",
    flags=regex.IGNORECASE,
)

def pre_tokenize(text: str) -> list[str]:
    return PATTERN.findall(text)


print(pre_tokenize("I don't code in Python 3.11!"))

['I', ' don', "'t", ' code', ' in', ' Python', ' 3', '.', '11', '!']


This code is responsible for pre-tokenization based on the standard GPT-2 pattern.

This algorithm splits the text into smaller chunks prior to BPE to prevent consecutive words or punctuation marks from merging:


In [11]:
# create pattern
# if have `regex`, use that but have not use `re`
try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )



## part 1: `def pre_tokenize`

To implement the `pre_tokenize` method, we must use `findall` with `GPT2_PATTERN` to extract all segments matching the regex rules as a list of strings.


In [12]:
# part 1----------------------------------------------
def pre_tokenize(text: str) -> List[str]:
    """
    Split input text into initial word/symbol chunks using the GPT-2 regex pattern.

    Args:
        text (str): Raw input text string to pre-tokenize.

    Returns:
        List[str]: A list of string chunks matched by the pre-tokenization regex.
    """
    # TODO: Apply GPT2_PATTERN regex iterator over text to extract all chunk string matches
    return GPT2_PATTERN.findall(text)


In [25]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
print(f"split form:---------------------\n{(sample_1.split())}")
print(f"use function pre_tokenize:------\n{pre_tokenize(sample_1)}")


------------------/1/-------------------
split form:---------------------
['I', 'am', 'Mohsen', 'Mohebbi.', 'I', 'participated', 'in', 'the', 'Daneshkar', 'Artificial', 'Intelligence', 'course.']
use function pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']
